# 1.9 參數微動時，代價會改多少？


代價高並未告訴我們哪個參數應該增大。先把文字表縮成一個能手算的旋鈕 w：希望它靠近 3，代價 `L=(w-3)²`。現在 w=1，L=4。

| w | 代價 L | 相對於 w=1 的改變 |
| --- | ---: | --- |
| 1 | 4 | 原位置 |
| 1.1 | 3.61 | w 增 0.1，L 減 0.39 |
| 1.01 | 3.9601 | w 增 0.01，L 減 0.0399 |

「代價改變÷參數改變」分別為 -3.9、-3.99。步子越小，比例越接近 -4；這個在目前位置的極限比例叫**導數**，表示附近的敏感度。對每個參數都算一個導數，再排回原位置，就叫**梯度**。

下面在左右各試很小一步：用 w+h 與 w-h 的代價差，除以兩點距離 2h。這種估算叫**有限差分**。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
def loss(w):
    return (w - 3) ** 2


w = 1.0
h = 0.001
left = loss(w - h)
right = loss(w + h)
gradient = (right - left) / (2 * h)
print(left, right)
print(gradient, "公式計算", 2 * (w - 3))

h=0.001 時，左點 0.999 的代價為 4.004001，右點 1.001 的代價為 3.996001。從左往右，代價減 0.008、w 增 0.002，相除得到 -4。平方代價的導數公式 `2*(w-3)` 也給 -4。

負號表示「現在稍微增加 w，代價會下降」，不是要把 w 改成 -4。大小 4 表示附近 w 增加 0.001 時，代價約減少 0.004。它描述微小改變，不能保證跳很遠也變好。

練習改 w=5：左側更靠近目標，梯度為 +4，所以希望往減少 w 的方向走。再試 w=3，兩側代價都約 0.000001，梯度為 0。程式只測敏感度，w 還沒有被更新。

<details>
<summary>補充與重做</summary>

函式和科學記號見 [W.2](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)、[W.4](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.4)；求導暖身見 [W.6](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)。h 太大可能不能代表附近，太小則兩個近似小數相減的誤差會被除法放大。大型模型不逐格試左右兩次，下一節會用運算關係算導數。

</details>
